<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day17_practice2_%EC%85%80%ED%94%84%EC%96%B4%ED%85%90%EC%85%98_%ED%95%B4%EB%B6%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 셀프 어텐션
# Q·K·V는 어디서 오나 : 임베딩×학습 행렬 (Linear 3개)

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, time
import matplotlib.pyplot as plt

torch.manual_seed(42)

In [5]:
# 셀 1. Q·K·V는 어디서 오나 - '같은 X'의 세 가지 변신
# 셀프(self) 어텐션: 질문도, 색인도, 내용도 전부 '같은 문장 X'에서 만든다, X = '영화 정말 재미있다'
# Q = X @ W_q / K = X @ W_k / V = X @ W_v
# W_q, W_k, W_v는 학습되는 행렬 (nn.Linear)

In [15]:
class SelfAttention(nn.Module):
  def __init__(self,dim):
    super().__init__()
    self.W_q = nn.Linear(dim, dim, bias = False) # Q·K·V를 만드는 학습 행렬 3개
    self.W_k = nn.Linear(dim, dim, bias = False)
    self.W_v = nn.Linear(dim, dim, bias = False)
    self.scale = math.sqrt(dim) # √d: 점수 나눌 값

  def forward(self, x): # x: (B, 단어수, dim)
    Q, K, V = self.W_q(x), self.W_k(x), self.W_v(x)
    # print(Q)
    # print(K)
    # print(V)
    attn = F.softmax(Q @ K.transpose(1,2) / self.scale, dim=1) # K.transpose(1,2): 전치 1번·2번 축을 맞바꿈(B,단어,dim)→(B,dim,단어)
    self.attn_map = attn # 시각화용 보관
    return attn @ V # 관련도 무게×내용

In [12]:
attn_layer = SelfAttention(dim=16)
x = torch.randn(1,3,16) # 단어 3개짜리 문장
out = attn_layer(x)
print("입력:", tuple(x.shape), "→ 출력:", tuple(out.shape), "(모양 유지 - 쌓기 좋다)")

tensor([[[ 0.1742,  1.1471, -0.5392, -0.6140,  0.5618,  1.0466, -0.4313,
          -0.2328,  0.5627,  0.3114,  0.0193,  0.0549,  0.9082,  0.3589,
          -0.0109, -0.4329],
         [ 0.3148, -0.6298,  0.0099, -0.3770, -0.1552, -0.3744, -0.2446,
          -0.3374, -0.0560,  0.3362,  0.6653,  0.7234,  0.0678,  0.6123,
          -0.4036,  0.4492],
         [-0.8046, -0.5761,  0.0655,  1.2859, -1.0880, -0.7580,  0.8712,
          -1.2950,  0.0734,  0.4735, -0.9146,  0.5403, -0.4466, -0.6481,
           0.0951, -0.0446]]], grad_fn=<UnsafeViewBackward0>)
tensor([[[-0.3905, -0.2005,  0.2533,  0.1274, -0.6987, -0.3683,  0.0295,
           0.3335,  0.5980,  0.1316,  0.1859,  0.0893,  0.0339,  0.3733,
           0.4261, -0.1821],
         [-0.7675,  0.7904, -0.8825, -1.2681, -0.1884,  0.6812,  0.3582,
          -0.1051, -0.6934,  0.4717, -0.1176,  0.3275,  0.3751,  0.2078,
          -0.1573,  0.5685],
         [ 0.1981,  0.5604,  0.5554, -0.5971,  0.0275, -0.2653,  0.9366,
           0.1642, 

In [16]:
def first_word_grad(layer, seq_len, dim=32): #RNN / LSTM / 어텐션
  torch.manual_seed(0)
  x = torch.randn(1, seq_len, dim, requires_grad=True)
  out = layer(x)
  if isinstance(out, tuple): out = out[0] # RNN/LSTM은 (out, h) 튜플, 어텐션은 텐서 하나
  out[0, -1].sum().backward() # 0번 문장의 마지막 단어 출력에서 역전파
  return x.grad[0,0].abs().mean().item() # 0번 문장 '첫 단어' 자리의 기울기
print("[첫 단어의 기울기 - 문장 길이 100]")
torch.manual_seed(0); print(f"RNN : {first_word_grad(nn.RNN(32, 32, batch_first=True), 100):.2e}")
torch.manual_seed(0); print(f"LSTM: {first_word_grad(nn.LSTM(32, 32, batch_first=True), 100):.2e}") # 잊기 게이트 0.5 열린 상태
torch.manual_seed(0); print(f"어텐션: {first_word_grad(SelfAttention(32), 100):.2e}")

[첫 단어의 기울기 - 문장 길이 100]
RNN : 1.08e-37
LSTM: 9.05e-24
어텐션: 3.86e-03
